# Assignment 7 — Abalone classification: logistic regression, PCA, SVM

**SVNIT — Department of Artificial Intelligence**

**Question sheet:** `Assignment-7.pdf`

**Dataset:** `../data/abalone.data` (UCI Abalone, 4,177 animals)

`Rings` is a count, not a category. The lab asks for classification, so rings are turned into three age groups before any model is trained. That rule is fixed in the code and explained where the target is built.

The same 80/20 split (`random_state=42`) is used for logistic regression and SVM, with and without PCA, so the comparison is fair. The scaler and the PCA are fit on the training set only.

Plots go to `../plots/assignment_7/`.


In [6]:
import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)

# UCI file has no header row. These names are the official column order.
COLUMNS = [
    "Sex",             # M, F, or I (infant)
    "Length",          # longest shell measurement (mm, already scaled in the file)
    "Diameter",
    "Height",
    "Whole_weight",    # whole animal
    "Shucked_weight",  # meat only
    "Viscera_weight",  # gut weight
    "Shell_weight",
    "Rings",           # +1.5 gives a rough age in years. This becomes the target.
]
NUM_COLS = [c for c in COLUMNS if c not in ("Sex", "Rings")]


def find_project_root():
    starts = [Path.cwd().resolve()]
    for key in ("__vsc_ipynb_file__", "__session__"):
        val = globals().get(key)
        if val:
            starts.append(Path(val).resolve().parent)
    seen = set()
    for start in starts:
        for p in [start, *start.parents]:
            if p in seen:
                continue
            seen.add(p)
            if (p / "data" / "abalone.data").exists():
                return p
    raise FileNotFoundError("Could not find data/abalone.data")


ROOT = find_project_root()
DATA = ROOT / "data"
PLOTS = ROOT / "plots" / "assignment_7"
PLOTS.mkdir(parents=True, exist_ok=True)


def section(title):
    print("\n" + "=" * 72)
    print(f"  {title}")
    print("=" * 72)


def save_plot(name):
    path = PLOTS / name
    plt.tight_layout()
    plt.savefig(path, dpi=120, bbox_inches="tight")
    plt.close()
    print(f"Saved {path.relative_to(ROOT)}")


def age_group(rings):
    """Turn the ring count into a class.

    Cuts sit on the quartiles of this file (25th ≈ 8, 50th ≈ 9, 75th ≈ 11),
    which also makes the three classes similar in size:
      young  : 8 or fewer rings
      medium : 9 or 10 rings
      old    : 11 or more rings
    Rings itself is NOT used as a feature. Otherwise the model would be
    predicting a label that was copied from one of its inputs.
    """
    if rings <= 8:
        return "young"
    if rings <= 10:
        return "medium"
    return "old"


def evaluate(name, model, X_test, y_test, labels):
    """Accuracy, confusion matrix, and the full per-class report."""
    pred = model.predict(X_test)
    acc = accuracy_score(y_test, pred)
    cm = confusion_matrix(y_test, pred, labels=labels)
    print(f"\n{name}")
    print(f"Test accuracy: {acc:.4f} ({100 * acc:.2f}%)")
    print("Confusion matrix (rows = true class, columns = predicted class)")
    print("Labels order:", labels)
    print(pd.DataFrame(cm, index=labels, columns=labels).to_string())
    print(classification_report(y_test, pred, labels=labels, digits=3))
    return acc, pred


print("Project root:", ROOT)


Project root: /Users/ojas/Desktop/ML Lab


## Part A — Inspect the table and make a class label


In [7]:
section("A1–A3: Load, EDA, encode the target")

abalone = pd.read_csv(DATA / "abalone.data", header=None, names=COLUMNS)
print(f"Rows: {len(abalone):,}    Columns: {abalone.shape[1]}")
print("Dtypes:")
print(abalone.dtypes.to_string())
print("\nMissing values:", int(abalone.isna().sum().sum()))
print("\nNumeric summary:")
print(abalone[NUM_COLS + ["Rings"]].describe().round(3).to_string())
print("\nSex counts (I = infant, not a third adult sex):")
print(abalone["Sex"].value_counts().to_string())

abalone["AgeGroup"] = abalone["Rings"].map(age_group)
# Fixed order so confusion matrices always read young / medium / old.
CLASS_ORDER = ["young", "medium", "old"]
print("\nAge groups created from Rings:")
print(abalone["AgeGroup"].value_counts().reindex(CLASS_ORDER).to_string())

# Distributions of the measurements the model will actually see.
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, col in zip(axes.ravel(), NUM_COLS):
    ax.hist(abalone[col], bins=30, color="steelblue", edgecolor="white")
    ax.set_title(col, fontsize=9)
fig.suptitle("Measurement distributions")
save_plot("eda_histograms.png")

# Length, diameter and the four weights move together. PCA will collapse
# that shared "overall size" direction into the first component.
plt.figure(figsize=(8, 6))
sns.heatmap(abalone[NUM_COLS].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation of the physical measurements")
save_plot("eda_correlation.png")

plt.figure(figsize=(8, 4.5))
sns.boxplot(data=abalone, x="AgeGroup", y="Shell_weight", order=CLASS_ORDER, color="steelblue")
plt.title("Shell weight rises with the age group")
save_plot("eda_shell_by_age.png")



  A1–A3: Load, EDA, encode the target
Rows: 4,177    Columns: 9
Dtypes:
Sex                   str
Length            float64
Diameter          float64
Height            float64
Whole_weight      float64
Shucked_weight    float64
Viscera_weight    float64
Shell_weight      float64
Rings               int64

Missing values: 0

Numeric summary:
          Length   Diameter     Height  Whole_weight  Shucked_weight  Viscera_weight  Shell_weight      Rings
count 4,177.0000 4,177.0000 4,177.0000    4,177.0000      4,177.0000      4,177.0000    4,177.0000 4,177.0000
mean      0.5240     0.4080     0.1400        0.8290          0.3590          0.1810        0.2390     9.9340
std       0.1200     0.0990     0.0420        0.4900          0.2220          0.1100        0.1390     3.2240
min       0.0750     0.0550     0.0000        0.0020          0.0010          0.0000        0.0020     1.0000
25%       0.4500     0.3500     0.1150        0.4420          0.1860          0.0940        0.1300     8.0

**EDA plots**

![Histograms](../plots/assignment_7/eda_histograms.png)

![Correlation](../plots/assignment_7/eda_correlation.png)

![Shell weight by age](../plots/assignment_7/eda_shell_by_age.png)


In [8]:
section("A4–A7: Split, scale, logistic regression")

# Features: sex + the 7 physical measurements. Rings is excluded on purpose.
X = abalone[["Sex"] + NUM_COLS]
y = abalone["AgeGroup"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {len(X_train):,}    Test: {len(X_test):,}")

# Sex is text. One-hot makes two 0/1 columns (drop='first' removes one of the
# three, because M+F+I would always equal 1 and PCA would see a useless direction).
# Unknown categories are not expected; the file only contains M, F, I.
sex_encoder = OneHotEncoder(drop="first", sparse_output=False)
sex_train = sex_encoder.fit_transform(X_train[["Sex"]])
sex_test = sex_encoder.transform(X_test[["Sex"]])
sex_names = list(sex_encoder.get_feature_names_out(["Sex"]))

# Glue the numeric measurements next to the sex dummies, THEN standardize.
# Standardization (zero mean, unit variance) is required before PCA and before
# logistic regression, because both treat a large column as a large effect.
# Fit on train, transform test. Test rows must not change the means.
X_train_raw = np.hstack([X_train[NUM_COLS].to_numpy(), sex_train])
X_test_raw = np.hstack([X_test[NUM_COLS].to_numpy(), sex_test])
feature_names = NUM_COLS + sex_names

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train_raw)
X_test_s = scaler.transform(X_test_raw)
print("Scaled feature matrix:", X_train_s.shape, "(rows, features)")
print("Feature order:", feature_names)

# Multinomial logistic regression: one linear score per class, then softmax
# turns the scores into probabilities that add up to 1.
# sklearn 1.8+ does this automatically for 3 classes (the old multi_class
# argument has been removed).
logit = LogisticRegression(max_iter=1000, random_state=42)
logit.fit(X_train_s, y_train)
acc_logit, pred_logit = evaluate(
    "Logistic regression on the 9 standardized features",
    logit, X_test_s, y_test, CLASS_ORDER,
)

plt.figure(figsize=(5.5, 4.5))
cm = confusion_matrix(y_test, pred_logit, labels=CLASS_ORDER)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_ORDER, yticklabels=CLASS_ORDER)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Logistic regression — test confusion matrix")
save_plot("logit_confusion.png")



  A4–A7: Split, scale, logistic regression
Train: 3,341    Test: 836
Scaled feature matrix: (3341, 9) (rows, features)
Feature order: ['Length', 'Diameter', 'Height', 'Whole_weight', 'Shucked_weight', 'Viscera_weight', 'Shell_weight', 'Sex_I', 'Sex_M']

Logistic regression on the 9 standardized features
Test accuracy: 0.6615 (66.15%)
Confusion matrix (rows = true class, columns = predicted class)
Labels order: ['young', 'medium', 'old']
        young  medium  old
young     224      46   11
medium     57     135   73
old        15      81  194
              precision    recall  f1-score   support

       young      0.757     0.797     0.776       281
      medium      0.515     0.509     0.512       265
         old      0.698     0.669     0.683       290

    accuracy                          0.661       836
   macro avg      0.657     0.659     0.657       836
weighted avg      0.660     0.661     0.660       836

Saved plots/assignment_7/logit_confusion.png


**Logistic regression confusion matrix**

![Logistic regression confusion matrix](../plots/assignment_7/logit_confusion.png)


In [9]:
section("A8–A12: PCA, then logistic regression on the components")

# PCA rotates the standardized features into new axes (components) that are
# uncorrelated. The first axis is the direction of greatest variance.
# explained_variance_ratio_[i] is the fraction of total variance on axis i.
pca_full = PCA(random_state=42)
pca_full.fit(X_train_s)
ratio = pca_full.explained_variance_ratio_
cumulative = np.cumsum(ratio)

print("explained_variance_ratio_:")
print(np.array2string(ratio, precision=4))
print(f"\n{'PC':<6} {'Variance share':>16} {'Cumulative':>12}")
print("-" * 36)
for i, (share, cum) in enumerate(zip(ratio, cumulative), start=1):
    print(f"PC{i:<4} {100 * share:>15.2f}% {100 * cum:>11.2f}%")

# "Selected" components = the smallest set that still keeps 95% of the variance.
# Fewer axes means a simpler model. The leftover 5% is mostly noise and
# duplicate size information (length and diameter are almost the same fact).
n_keep = int(np.searchsorted(cumulative, 0.95) + 1)
print(f"\nComponents kept for 95% variance: {n_keep} of {len(ratio)}")

pca = PCA(n_components=n_keep, random_state=42)
X_train_pca = pca.fit_transform(X_train_s)
X_test_pca = pca.transform(X_test_s)

logit_pca = LogisticRegression(max_iter=1000, random_state=42)
logit_pca.fit(X_train_pca, y_train)
acc_logit_pca, pred_logit_pca = evaluate(
    f"Logistic regression on the first {n_keep} principal components",
    logit_pca, X_test_pca, y_test, CLASS_ORDER,
)

# coef_ has shape (n_classes, n_components).
# Entry [class, k] is the weight of PCk in that class's log-odds.
# A large positive weight means "this component pushes the prediction toward this class".
coef = pd.DataFrame(
    logit_pca.coef_,
    index=logit_pca.classes_,
    columns=[f"PC{i}" for i in range(1, n_keep + 1)],
)
coef = coef.reindex(CLASS_ORDER)
print("\nLogistic coefficients on the principal components:")
print(coef.round(3).to_string())

plt.figure(figsize=(8, 3.2))
sns.heatmap(coef, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Logistic regression weights on each principal component")
plt.ylabel("Class")
save_plot("pca_coefficient_heatmap.png")

plt.figure(figsize=(5.5, 4.5))
cm = confusion_matrix(y_test, pred_logit_pca, labels=CLASS_ORDER)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_ORDER, yticklabels=CLASS_ORDER)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("PCA + logistic regression — test confusion matrix")
save_plot("logit_pca_confusion.png")



  A8–A12: PCA, then logistic regression on the components
explained_variance_ratio_:
[7.6750e-01 1.2815e-01 4.4506e-02 2.3346e-02 1.6174e-02 1.1452e-02
 6.8404e-03 1.3111e-03 7.1677e-04]

PC       Variance share   Cumulative
------------------------------------
PC1              76.75%       76.75%
PC2              12.81%       89.57%
PC3               4.45%       94.02%
PC4               2.33%       96.35%
PC5               1.62%       97.97%
PC6               1.15%       99.11%
PC7               0.68%       99.80%
PC8               0.13%       99.93%
PC9               0.07%      100.00%

Components kept for 95% variance: 4 of 9

Logistic regression on the first 4 principal components
Test accuracy: 0.5969 (59.69%)
Confusion matrix (rows = true class, columns = predicted class)
Labels order: ['young', 'medium', 'old']
        young  medium  old
young     213      54   14
medium     60      88  117
old        23      69  198
              precision    recall  f1-score   support

      

**PCA logistic regression**

![Coefficient heatmap](../plots/assignment_7/pca_coefficient_heatmap.png)

![PCA logistic confusion matrix](../plots/assignment_7/logit_pca_confusion.png)


## Part B — Support vector machine

An SVM looks for the boundary with the widest margin between classes. The RBF kernel lets that boundary curve, which a plain logistic regression (a flat boundary in feature space) cannot do. The same split, the same scaler, and the same principal components as Part A are reused.


In [10]:
section("Part B: SVM, then the comparison")

# Documented settings:
#   kernel='rbf'  — compare each animal with others using a Gaussian similarity.
#   C=1.0         — default penalty on points that fall on the wrong side.
#                   Larger C tries harder to classify every training point.
#   gamma='scale' — kernel width = 1 / (n_features * variance). sklearn's default.
svm = SVC(kernel="rbf", C=1.0, gamma="scale", random_state=42)
svm.fit(X_train_s, y_train)
acc_svm, pred_svm = evaluate(
    "SVM (RBF, C=1, gamma='scale') on the 9 standardized features",
    svm, X_test_s, y_test, CLASS_ORDER,
)

svm_pca = SVC(kernel="rbf", C=1.0, gamma="scale", random_state=42)
svm_pca.fit(X_train_pca, y_train)
acc_svm_pca, pred_svm_pca = evaluate(
    f"SVM (same kernel) on the first {n_keep} principal components",
    svm_pca, X_test_pca, y_test, CLASS_ORDER,
)

comparison = pd.DataFrame([
    ("Logistic regression", "standardized features", acc_logit),
    ("Logistic regression", f"first {n_keep} PCs", acc_logit_pca),
    ("SVM (RBF)", "standardized features", acc_svm),
    ("SVM (RBF)", f"first {n_keep} PCs", acc_svm_pca),
], columns=["Model", "Features", "Test accuracy"])

print("\nSame split, four models")
print(f"{'Model':<24} {'Features':<28} {'Test accuracy':>14}")
print("-" * 68)
for _, row in comparison.iterrows():
    print(f"{row['Model']:<24} {row['Features']:<28} {100 * row['Test accuracy']:>13.2f}%")

best = comparison.loc[comparison["Test accuracy"].idxmax()]
print(
    f"\nBest on this test set: {best['Model']} using {best['Features']} "
    f"({100 * best['Test accuracy']:.2f}%)."
)
print("PCA usually costs a little accuracy, because the dropped components")
print("still held some age signal, and it buys a smaller, uncorrelated input.")
print("The RBF SVM can bend the boundary, so it is the model to beat the")
print("straight logistic boundary when the classes overlap in size.")

plt.figure(figsize=(8, 4.5))
labels = [f"{m}\n{f}" for m, f in zip(comparison["Model"], comparison["Features"])]
plt.bar(labels, 100 * comparison["Test accuracy"], color=["steelblue", "skyblue", "seagreen", "darkseagreen"])
plt.ylabel("Test accuracy (%)")
plt.ylim(0, 100)
plt.title("Same split: logistic regression vs SVM, with and without PCA")
save_plot("model_comparison.png")

plt.figure(figsize=(5.5, 4.5))
cm = confusion_matrix(y_test, pred_svm, labels=CLASS_ORDER)
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens",
            xticklabels=CLASS_ORDER, yticklabels=CLASS_ORDER)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("SVM — test confusion matrix")
save_plot("svm_confusion.png")



  Part B: SVM, then the comparison

SVM (RBF, C=1, gamma='scale') on the 9 standardized features
Test accuracy: 0.6758 (67.58%)
Confusion matrix (rows = true class, columns = predicted class)
Labels order: ['young', 'medium', 'old']
        young  medium  old
young     214      60    7
medium     52     139   74
old        12      66  212
              precision    recall  f1-score   support

       young      0.770     0.762     0.766       281
      medium      0.525     0.525     0.525       265
         old      0.724     0.731     0.727       290

    accuracy                          0.676       836
   macro avg      0.673     0.672     0.672       836
weighted avg      0.676     0.676     0.676       836


SVM (same kernel) on the first 4 principal components
Test accuracy: 0.6232 (62.32%)
Confusion matrix (rows = true class, columns = predicted class)
Labels order: ['young', 'medium', 'old']
        young  medium  old
young     199      68   14
medium     48     124   93
old  

**SVM and the four-way comparison**

![Model comparison](../plots/assignment_7/model_comparison.png)

![SVM confusion matrix](../plots/assignment_7/svm_confusion.png)
